# DineSense v3: fine-tuning a pretrained transformer

This notebook fine-tunes **DistilRoBERTa** to label restaurant reviews as negative, neutral or positive. It uses exactly the same train / validation / test split as your baseline and BiLSTMs, so the scores are directly comparable.

| Model | Test macro F1 |
|---|---|
| TF-IDF + logistic regression | 0.772 |
| BiLSTM v1 | 0.748 |
| BiLSTM v2 (frozen GloVe) | 0.754 |
| **DistilRoBERTa v3** | *this notebook* |

**Before you start**
1. In Colab: *Runtime → Change runtime type → T4 GPU → Save*.
2. Have these two files from your laptop ready to upload:
   - `D:\DineSense\data\processed\restaurant_reviews.csv`
   - `D:\DineSense\ml\preprocessing.py`
3. Run the cells in order (*Runtime → Run all* also works). A full run takes roughly 20–40 minutes on a T4.

**The idea.** Your BiLSTM had to learn English from 42,000 reviews. DistilRoBERTa has already been trained on billions of words of web text, so it arrives knowing grammar, negation and how context changes meaning. Fine-tuning adds one small new layer that maps the model's reading of a review to your three labels, then trains the whole network for a few epochs with a *small* learning rate. It adapts to restaurant reviews without forgetting what it knew. Think of starting a geometry optimisation from a structure that is already close to the minimum: small steps, and not many of them.

## 0. Settings
These are the only values you should need to change.

In [ ]:
# ---- Settings ----
MODEL_NAME = "distilbert/distilroberta-base"  # pretrained model from the Hugging Face Hub
MAX_LEN = 256           # tokens per review; longer reviews are cut off at the end
BATCH_SIZE = 32         # lower to 16 if you get a CUDA out-of-memory error
EPOCHS = 3              # passes over the training set
LEARNING_RATE = 2e-5    # small on purpose: we are refining, not starting from scratch
WEIGHT_DECAY = 0.01     # mild regularisation on the weights
WARMUP_FRACTION = 0.06  # ramp the learning rate up over the first 6% of steps
EVALS_PER_EPOCH = 2     # score the validation set twice per epoch
SEED = 42

SMOKE_TEST = False      # True = train on a few hundred reviews to check everything runs (~2 min)
ALLOW_CPU = False       # leave False on Colab; training on a CPU would take many hours

CSV_NAME = "restaurant_reviews.csv"
OUT_DIR = "transformer_v3"      # final model, metrics and plots go here
BEST_DIR = "best_checkpoint"    # best model so far, updated during training

In [ ]:
import os, sys, platform, random, warnings
import numpy as np
import pandas as pd
import torch
import transformers

print("Python      ", platform.python_version())
print("PyTorch     ", torch.__version__)
print("Transformers", transformers.__version__)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU         ", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    if not ALLOW_CPU:
        raise RuntimeError(
            "No GPU found. In Colab: Runtime -> Change runtime type -> T4 GPU -> Save, "
            "then run the notebook again from the top."
        )
    print("Running on CPU (ALLOW_CPU=True). This will be slow.")

USE_AMP = device.type == "cuda"   # mixed precision: faster on a GPU, same accuracy
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")

## 1. Upload your data
Run the cell below. If the two files aren't already in Colab's Files panel, an upload button appears: select both `restaurant_reviews.csv` and `preprocessing.py`. The CSV is about 42 MB.

Files uploaded this way are deleted when the Colab runtime disconnects, so you may need to upload them again in a later session.

In [ ]:
from pathlib import Path

needed = [CSV_NAME, "preprocessing.py"]
missing = [f for f in needed if not Path(f).exists()]
if missing:
    try:
        from google.colab import files
        print("Select these files:", ", ".join(missing))
        files.upload()
    except ImportError:
        pass

missing = [f for f in needed if not Path(f).exists()]
if missing:
    raise FileNotFoundError(
        f"Still missing: {missing}. Drag them into the Files panel on the left, then run this cell again."
    )
print("Found:", ", ".join(f"{f} ({Path(f).stat().st_size / 1e6:.1f} MB)" for f in needed))

## 2. Recreate your exact split
This calls your own `load_dataset()` and `split_dataset()` from `preprocessing.py`, with seed 42. The test set has to be the same 8,996 reviews your other models were scored on, or the comparison means nothing.

One deliberate difference: the BiLSTM read your *cleaned*, lowercased text. The transformer reads the original review, because its tokenizer was built for real-world writing, and capital letters, emoji and punctuation carry tone it can use.

In [ ]:
import html, re

sys.path.insert(0, os.getcwd())
from preprocessing import LABELS, load_dataset, split_dataset

df = load_dataset(CSV_NAME)
train_df, val_df, test_df = split_dataset(df)

if len(test_df) == 8996:
    print("Test set matches the one your other models used (8,996 reviews).")
else:
    print(f"WARNING: the test set has {len(test_df):,} reviews, not 8,996. Is this the same CSV "
          "your other models used? If not, the scores won't be comparable.")

_ws = re.compile(r"\s+")
def raw_text(t):
    """Light cleanup only: decode HTML entities and collapse whitespace."""
    return _ws.sub(" ", html.unescape(str(t))).strip()

for part in (train_df, val_df, test_df):
    part["text"] = part["review_text"].map(raw_text)

if SMOKE_TEST:
    train_df = train_df.sample(600, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(200, random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(200, random_state=SEED).reset_index(drop=True)
    print(f"SMOKE_TEST: using {len(train_df)} / {len(val_df)} / {len(test_df)} reviews")

print(train_df["sentiment"].value_counts().to_string())

## 3. Tokenize: how the model reads text
The model doesn't see words. Its tokenizer splits text into *subword* pieces from a fixed vocabulary of about 50,000. Common words stay whole and rare words are built from pieces, so no word is ever unknown. That fixes a problem GloVe had: words like "kottu" or "deliciousss" got no useful vector.

In the output, `Ġ` marks a token that starts with a space. Each review becomes a list of token IDs, cut off at `MAX_LEN`. The cell also checks how many reviews lose their ending. You may see a warning about sequences longer than 512; it comes from this length check and is harmless.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

for s in ["The food was not bad at all!", "Best kottu in town, sooo good", "Waited 45 mins. Never again."]:
    print(f"{s!r}\n   -> {tokenizer.tokenize(s)}\n")

sample = train_df["text"].sample(min(3000, len(train_df)), random_state=SEED).tolist()
lengths = np.array([len(ids) for ids in tokenizer(sample, truncation=False)["input_ids"]])
print(f"Tokens per review: median {np.median(lengths):.0f}, 90th percentile {np.percentile(lengths, 90):.0f}")
print(f"Reviews longer than MAX_LEN={MAX_LEN}: {np.mean(lengths > MAX_LEN):.1%} (these lose their ending)")

## 4. Load the pretrained model
`AutoModelForSequenceClassification` loads DistilRoBERTa's pretrained body (6 transformer layers) and puts a small **classification head** on top with 3 outputs, one per label. The body is pretrained; the head starts random.

You'll see a load report. Weights marked **MISSING** are the new head, which starts random. Weights marked **UNEXPECTED** belong to the part of the original model used for its pretraining task, which you don't need. Both are expected.

In [ ]:
from transformers import AutoModelForSequenceClassification

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
id2label = {i: label for i, label in enumerate(LABELS)}
label2id = {label: i for i, label in enumerate(LABELS)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)
model.float().to(device)   # keep master weights in float32; mixed precision is applied per step

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params / 1e6:.1f}M weights, all trainable")

## 5. Batches
A DataLoader serves the reviews in shuffled mini-batches. Each batch is tokenized on the fly and padded only to the longest review *in that batch*. Most reviews are far shorter than 256 tokens, so this saves a lot of time.

In [ ]:
from torch.utils.data import DataLoader

def make_loader(frame, shuffle):
    pairs = list(zip(frame["text"].tolist(), frame["label"].tolist()))

    def collate(batch):
        texts, labels = zip(*batch)
        enc = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"], torch.tensor(labels)

    return DataLoader(
        pairs,
        batch_size=BATCH_SIZE if shuffle else BATCH_SIZE * 2,   # no gradients at eval time, so bigger batches fit
        shuffle=shuffle,
        collate_fn=collate,
        generator=torch.Generator().manual_seed(SEED),
    )

train_loader = make_loader(train_df, shuffle=True)
val_loader = make_loader(val_df, shuffle=False)
test_loader = make_loader(test_df, shuffle=False)
print(f"{len(train_loader)} training batches per epoch")

## 6. Optimiser and learning-rate schedule
- **AdamW** is Adam with weight decay handled correctly. It's the standard optimiser for fine-tuning.
- **A learning rate of 2e-5** is 50 times smaller than the 1e-3 your BiLSTM used. The pretrained weights already sit near a good minimum, and big steps would knock them out of it, so the model would forget what it knew.
- **Warmup, then linear decay.** The rate climbs from 0 to 2e-5 over the first 6% of steps, then falls steadily to 0. Warmup protects the pretrained weights while the random head is still producing large, noisy gradients.
- Biases and LayerNorm weights are left out of weight decay. That's standard practice: shrinking them towards zero doesn't regularise anything useful.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
param_groups = [
    {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)],
     "weight_decay": 0.0},
]
optimizer = AdamW(param_groups, lr=LEARNING_RATE)

total_steps = EPOCHS * len(train_loader)
warmup_steps = max(1, int(WARMUP_FRACTION * total_steps))

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

scheduler = LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"{total_steps} optimiser steps in total, {warmup_steps} of them warmup")

## 7. Train
This is the loop you stepped through on the Deep Learning Bench page: forward pass, loss, `zero_grad`, `backward`, `step`. There are three additions:
- **Mixed precision** (`autocast` and `GradScaler`). Most of the maths runs in 16-bit on the GPU, roughly doubling the speed. The scaler stops very small gradients from rounding to zero.
- **Gradient clipping** caps the size of any single update, so one unusual batch can't throw the weights off.
- **Checkpointing on validation macro F1.** Twice per epoch the model is scored on the validation set, and whenever it beats its best so far it's saved. If later steps overfit, you still keep the best version.

Watch the printed validation scores. Validation loss rising while training loss keeps falling is overfitting, the same shape as your BiLSTM v1. With a pretrained model this often shows up in epoch 3, which is why the best checkpoint is kept.

In [ ]:
import time, json
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, n, preds, golds = 0.0, 0, [], []
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        total_loss += F.cross_entropy(logits.float(), labels, reduction="sum").item()
        n += labels.size(0)
        preds.append(logits.argmax(-1).cpu())
        golds.append(labels.cpu())
    model.train()
    preds, golds = torch.cat(preds).numpy(), torch.cat(golds).numpy()
    return {"loss": total_loss / n, "accuracy": accuracy_score(golds, preds),
            "macro_f1": f1_score(golds, preds, average="macro"), "preds": preds, "golds": golds}

steps_per_epoch = len(train_loader)
eval_points = sorted({int(steps_per_epoch * k / EVALS_PER_EPOCH) for k in range(1, EVALS_PER_EPOCH + 1)})
history = {"epoch": [], "train_loss": [], "val_loss": [], "val_accuracy": [], "val_macro_f1": [], "learning_rate": []}
step_losses = []
best_f1, best_epoch = -1.0, None
start = time.time()
model.train()

for epoch in range(1, EPOCHS + 1):
    running, count = 0.0, 0
    bar = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")
    for i, (input_ids, attention_mask, labels) in enumerate(bar, start=1):
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits   # forward pass
        loss = F.cross_entropy(logits.float(), labels)                                    # how wrong?

        optimizer.zero_grad(set_to_none=True)                     # clear old gradients
        scaler.scale(loss).backward()                             # backpropagation
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # cap the update size
        scaler.step(optimizer)                                    # every weight takes a small step downhill
        scaler.update()
        scheduler.step()                                          # adjust the learning rate

        running += loss.item()
        count += 1
        step_losses.append(loss.item())
        bar.set_postfix(loss=f"{running / count:.3f}", lr=f"{scheduler.get_last_lr()[0]:.1e}")

        if i in eval_points:
            v = evaluate(val_loader)
            at = round(epoch - 1 + i / steps_per_epoch, 2)
            history["epoch"].append(at)
            history["train_loss"].append(running / count)
            history["val_loss"].append(v["loss"])
            history["val_accuracy"].append(v["accuracy"])
            history["val_macro_f1"].append(v["macro_f1"])
            history["learning_rate"].append(scheduler.get_last_lr()[0])
            improved = v["macro_f1"] > best_f1
            if improved:
                best_f1, best_epoch = v["macro_f1"], at
                model.save_pretrained(BEST_DIR)
                tokenizer.save_pretrained(BEST_DIR)
            tqdm.write(f"  epoch {at:.2f} | train loss {running / count:.3f} | val loss {v['loss']:.3f} | "
                       f"val acc {v['accuracy']:.3f} | val macro F1 {v['macro_f1']:.4f}"
                       + ("  <- best so far, saved" if improved else ""))
            running, count = 0.0, 0

print(f"Finished in {(time.time() - start) / 60:.1f} min. "
      f"Best validation macro F1 {best_f1:.4f} at epoch {best_epoch}.")

### Training curves
The left plot should look like the good-fit shape from chapter 7 of the Bench page. The dashed lines on the right are your other models' *test* scores, shown for rough reference only; the real comparison is in step 8.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
w = max(1, steps_per_epoch // 20)
smooth = np.convolve(step_losses, np.ones(w) / w, mode="valid")
ax1.plot((np.arange(len(smooth)) + w) / steps_per_epoch, smooth, alpha=0.8, label="training loss (smoothed)")
ax1.plot(history["epoch"], history["val_loss"], "o-", label="validation loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("cross-entropy loss"); ax1.set_title("Loss"); ax1.legend()

ax2.plot(history["epoch"], history["val_macro_f1"], "o-", color="C2", label="DistilRoBERTa (validation)")
ax2.axhline(0.7718, ls="--", color="gray", label="baseline (test) 0.772")
ax2.axhline(0.7544, ls=":", color="gray", label="BiLSTM v2 (test) 0.754")
ax2.set_xlabel("epoch"); ax2.set_ylabel("macro F1"); ax2.set_title("Validation macro F1"); ax2.legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## 8. Score on the test set, once
The test set has been locked away until now. This loads the best checkpoint and scores it a single time. If you keep changing settings and re-checking the test score, you slowly tune to the test set and the number stops meaning anything, so make decisions from validation scores.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).float().to(device)
t = evaluate(test_loader)
report = classification_report(t["golds"], t["preds"], labels=list(range(len(LABELS))),
                               target_names=LABELS, output_dict=True, zero_division=0)
print(classification_report(t["golds"], t["preds"], labels=list(range(len(LABELS))),
                            target_names=LABELS, digits=3, zero_division=0))

print("Test macro F1, same 8,996 reviews" if not SMOKE_TEST else "Test macro F1 (SMOKE_TEST: not comparable)")
for name, f1 in [("TF-IDF + logistic regression", 0.7718), ("BiLSTM v1", 0.7483),
                 ("BiLSTM v2 (GloVe)", 0.7544), ("DistilRoBERTa v3", t["macro_f1"])]:
    print(f"  {name:<30} {f1:.4f}")

cm = confusion_matrix(t["golds"], t["preds"], labels=list(range(len(LABELS))))
cm_pct = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
for r in range(len(LABELS)):
    for c in range(len(LABELS)):
        ax.text(c, r, f"{cm[r, c]}\n{cm_pct[r, c]:.1f}%", ha="center", va="center",
                color="white" if cm_pct[r, c] > 55 else "black")
ax.set_xticks(range(len(LABELS)), LABELS)
ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("DistilRoBERTa v3 (test set)")
plt.tight_layout()
plt.savefig("confusion_transformer_v3.png", dpi=150)
plt.show()

## 9. Read the mistakes
The scores tell you how often the model is wrong; reading the errors tells you why. Check whether the wrong answers are defensible. A 3-star review that reads as clearly positive is a label problem, not a model problem. Note the patterns you see for your report.

In [ ]:
errs = test_df.assign(pred=[LABELS[p] for p in t["preds"]])
errs = errs[errs["pred"] != errs["sentiment"]]
print(f"{len(errs):,} of {len(test_df):,} test reviews misclassified\n")
print(pd.crosstab(errs["sentiment"], errs["pred"], rownames=["true"], colnames=["predicted"]), "\n")
for _, r in errs.sample(min(12, len(errs)), random_state=1).iterrows():
    print(f"[{r['star_rating']} stars | true {r['sentiment']} | predicted {r['pred']}]")
    print("  " + r["text"][:350] + ("..." if len(r["text"]) > 350 else "") + "\n")

## 10. Save and download
This puts the model, tokenizer, metrics and plots in one folder and zips it (about 300 MB). Saving to Google Drive is the most reliable way to get a file this size out of Colab. A direct browser download also works, but it can stall.

On your laptop, unzip it so the files end up in `D:\DineSense\ml\saved_models\transformer_v3\`.

In [ ]:
import shutil

out = Path(OUT_DIR)
if out.exists():
    shutil.rmtree(out)
shutil.copytree(BEST_DIR, out)

metrics = {
    "model": "distilroberta_finetuned",
    "model_version": "v3",
    "base_model": MODEL_NAME,
    "test_accuracy": float(t["accuracy"]),
    "test_macro_f1": float(t["macro_f1"]),
    "per_class": {label: {k: float(v) for k, v in report[label].items()} for label in LABELS},
    "best_epoch": best_epoch,
    "epochs_run": EPOCHS,
    "best_val_macro_f1": float(best_f1),
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "trainable_params": int(n_params),
    "n_train": len(train_df),
    "n_test": len(test_df),
    "smoke_test": SMOKE_TEST,
    "trained_on": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
}
(out / "transformer_v3_metrics.json").write_text(json.dumps(metrics, indent=2))
(out / "transformer_v3_history.json").write_text(json.dumps(history, indent=2))
for png in ("training_curves.png", "confusion_transformer_v3.png"):
    if Path(png).exists():
        shutil.copy(png, out / png)

zip_path = shutil.make_archive(OUT_DIR, "zip", root_dir=".", base_dir=OUT_DIR)
print(f"Wrote {zip_path} ({Path(zip_path).stat().st_size / 1e6:.0f} MB)")
print("Contents:", ", ".join(sorted(p.name for p in out.iterdir())))

In [ ]:
SAVE_TO_DRIVE = True   # False = download straight to your browser instead

try:
    from google.colab import drive, files
    if SAVE_TO_DRIVE:
        drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive/DineSense")
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy(zip_path, dest / Path(zip_path).name)
        print(f"Saved to Google Drive: MyDrive/DineSense/{Path(zip_path).name}")
    else:
        files.download(zip_path)
except ImportError:
    print("Not running in Colab. The zip is at", Path(zip_path).resolve())

## If something goes wrong
- **CUDA out of memory:** set `BATCH_SIZE = 16` in the settings cell, then *Runtime → Restart session* and run everything again.
- **"No GPU found":** *Runtime → Change runtime type → T4 GPU*. If Colab has no GPUs free, try later, or use a Kaggle notebook (*Settings → Accelerator → GPU T4 x1*).
- **The runtime disconnected during training:** free Colab sessions stop if the tab sits idle too long. Keep the tab open; a full run fits comfortably inside the time limit.
- **The test-set warning in step 2:** you uploaded a different CSV from the one your other models used.
- **Want a dry run first?** Set `SMOKE_TEST = True` and run everything. It finishes in a few minutes with meaningless scores, but you'll know every step works. Then set it back to `False` and run again.

## Next
Once the `transformer_v3` folder is in `ml/saved_models`, add it to `MODEL_REGISTRY` in `ml/predict.py` so the app can serve it.